# Interview Transcriber — launcher

Use a T4 GPU runtime and the same Google account whose Drive contains the recordings. Store the Hugging Face read-only token once as `HF_TOKEN` in Colab Secrets. Run the single START cell below. It updates the local Git checkout first and executes the bootstrap directly from that checkout, avoiding stale GitHub raw/CDN copies.

In [ ]:
# @title ▶ START INTERVIEW TRANSCRIBER
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/playply/transcriber.git"
REPO_DIR = Path("/content/transcriber")

print("Updating Interview Transcriber from GitHub...", flush=True)
if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", "main"],
        check=True,
        timeout=60,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
        check=True,
        stdout=subprocess.DEVNULL,
        timeout=30,
    )
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
        timeout=90,
    )

head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"✓ Local repository updated ({head})", flush=True)

bootstrap_path = REPO_DIR / "colab_bootstrap.py"
source = bootstrap_path.read_text(encoding="utf-8")
if "LAUNCHER_BUILD" not in source:
    raise RuntimeError("Local colab_bootstrap.py is invalid.")

exec(compile(source, str(bootstrap_path), "exec"), {"__name__": "__main__"})
